In [ ]:
"""
image_processing_oop.py

Full Object-Oriented Image Processing Toolkit
Implements core concepts from scratch:

- Sampling and Quantization
- Noise models (Gaussian, Salt-Pepper, Poisson)
- Linear filters (Average, Gaussian)
- Non-linear filters (Median)
- Correlation and Convolution
- Sobel Edge Detection
- Full Canny Edge Detection

Author: Your Name
"""

import numpy as np
import matplotlib.pyplot as plt


class ImageProcessor:
    """
    Core image processing class implementing fundamental operations.
    All filters implemented from scratch using NumPy.
    """

    def __init__(self, image: np.ndarray):
        """
        Initialize processor with grayscale image.

        Args:
            image (np.ndarray): Input grayscale image
        """
        self.image = image.astype(float)

    # =============================
    # Basic Utilities
    # =============================

    @staticmethod
    def display(image: np.ndarray, title: str = "Image") -> None:
        """Display grayscale image."""
        plt.imshow(image, cmap='gray')
        plt.title(title)
        plt.axis('off')
        plt.show()

    def downsample(self, factor: int) -> np.ndarray:
        """Reduce sampling resolution."""
        return self.image[::factor, ::factor]

    def quantize(self, levels: int) -> np.ndarray:
        """Reduce intensity levels."""
        step = 256 / levels
        return np.floor(self.image / step) * step

    # =============================
    # Noise Models
    # =============================

    def add_gaussian_noise(self, mean=0, std=20) -> np.ndarray:
        """Add Gaussian noise."""
        noise = np.random.normal(mean, std, self.image.shape)
        return self.image + noise

    def add_salt_pepper_noise(self, prob=0.02) -> np.ndarray:
        """Add salt and pepper noise."""
        noisy = self.image.copy()
        num_pixels = int(prob * self.image.size)

        coords = np.random.randint(0, self.image.shape[0], (num_pixels, 2))
        for coord in coords:
            noisy[coord[0], coord[1]] = 255

        coords = np.random.randint(0, self.image.shape[0], (num_pixels, 2))
        for coord in coords:
            noisy[coord[0], coord[1]] = 0

        return noisy

    def add_poisson_noise(self) -> np.ndarray:
        """Add Poisson noise."""
        normalized = self.image / 255.0
        noisy = np.random.poisson(normalized * 255) / 255.0
        return noisy * 255

    # =============================
    # Kernel Operations
    # =============================

    @staticmethod
    def flip_kernel(kernel: np.ndarray) -> np.ndarray:
        """Flip kernel for convolution."""
        return np.flip(kernel)

    def correlate(self, kernel: np.ndarray) -> np.ndarray:
        """Perform correlation."""
        h, w = self.image.shape
        kh, kw = kernel.shape

        output = np.zeros((h - kh + 1, w - kw + 1))

        for i in range(h - kh + 1):
            for j in range(w - kw + 1):
                region = self.image[i:i+kh, j:j+kw]
                output[i, j] = np.sum(region * kernel)

        return output

    def convolve(self, kernel: np.ndarray) -> np.ndarray:
        """Perform convolution (kernel flipped)."""
        flipped = self.flip_kernel(kernel)
        return self.correlate(flipped)

    # =============================
    # Linear Filters
    # =============================

    def average_filter(self) -> np.ndarray:
        """Apply average filter."""
        kernel = np.ones((3, 3)) / 9
        return self.convolve(kernel)

    def gaussian_filter(self) -> np.ndarray:
        """Apply Gaussian filter."""
        kernel = np.array([
            [1, 2, 1],
            [2, 4, 2],
            [1, 2, 1]
        ]) / 16
        return self.convolve(kernel)

    # =============================
    # Non-linear Filters
    # =============================

    def median_filter(self) -> np.ndarray:
        """Apply median filter."""
        h, w = self.image.shape
        output = np.zeros((h - 2, w - 2))

        for i in range(h - 2):
            for j in range(w - 2):
                region = self.image[i:i+3, j:j+3]
                output[i, j] = np.median(region)

        return output

    # =============================
    # Sobel Edge Detection
    # =============================

    def sobel(self):
        """Compute Sobel gradient magnitude and direction."""

        gx_kernel = np.array([
            [-1, 0, 1],
            [-2, 0, 2],
            [-1, 0, 1]
        ])

        gy_kernel = np.array([
            [-1, -2, -1],
            [0, 0, 0],
            [1, 2, 1]
        ])

        grad_x = self.convolve(gx_kernel)
        grad_y = self.convolve(gy_kernel)

        magnitude = np.sqrt(grad_x**2 + grad_y**2)
        direction = np.arctan2(grad_y, grad_x)

        return magnitude, direction

    # =============================
    # Canny Edge Detection
    # =============================

    def non_max_suppression(self, mag, direction):
        """Perform Non-Maximum Suppression."""
        h, w = mag.shape
        output = np.zeros((h, w))

        angle = direction * 180 / np.pi
        angle[angle < 0] += 180

        for i in range(1, h-1):
            for j in range(1, w-1):

                q = 255
                r = 255

                if (0 <= angle[i, j] < 22.5) or (157.5 <= angle[i, j] <= 180):
                    q = mag[i, j+1]
                    r = mag[i, j-1]

                elif 22.5 <= angle[i, j] < 67.5:
                    q = mag[i+1, j-1]
                    r = mag[i-1, j+1]

                elif 67.5 <= angle[i, j] < 112.5:
                    q = mag[i+1, j]
                    r = mag[i-1, j]

                elif 112.5 <= angle[i, j] < 157.5:
                    q = mag[i-1, j-1]
                    r = mag[i+1, j+1]

                if mag[i, j] >= q and mag[i, j] >= r:
                    output[i, j] = mag[i, j]

        return output

    def double_threshold(self, image, low, high):
        """Apply double threshold."""
        strong = 255
        weak = 75

        output = np.zeros(image.shape)

        strong_i, strong_j = np.where(image >= high)
        weak_i, weak_j = np.where((image >= low) & (image < high))

        output[strong_i, strong_j] = strong
        output[weak_i, weak_j] = weak

        return output, weak, strong

    def hysteresis(self, image, weak, strong):
        """Perform edge tracking by hysteresis."""
        h, w = image.shape

        for i in range(1, h-1):
            for j in range(1, w-1):
                if image[i, j] == weak:

                    if np.any(image[i-1:i+2, j-1:j+2] == strong):
                        image[i, j] = strong
                    else:
                        image[i, j] = 0

        return image

    def canny(self):
        """Full Canny Edge Detection."""
        smoothed = self.gaussian_filter()

        processor = ImageProcessor(smoothed)
        mag, direction = processor.sobel()

        nms = self.non_max_suppression(mag, direction)

        thresholded, weak, strong = self.double_threshold(nms, 20, 50)

        final = self.hysteresis(thresholded, weak, strong)

        return final


# =============================
# Demo Runner
# =============================

class Demo:
    """Demonstration class."""

    @staticmethod
    def run():
        """Run all demonstrations."""

        image = np.zeros((200, 200))
        image[:, :100] = 50
        image[:, 100:] = 200

        processor = ImageProcessor(image)

        processor.display(image, "Original")

        gaussian_noisy = processor.add_gaussian_noise()
        processor.display(gaussian_noisy, "Gaussian Noise")

        sp_noisy = processor.add_salt_pepper_noise()
        processor.display(sp_noisy, "Salt & Pepper Noise")

        gaussian_filtered = processor.gaussian_filter()
        processor.display(gaussian_filtered, "Gaussian Filter")

        median_filtered = processor.median_filter()
        processor.display(median_filtered, "Median Filter")

        sobel_mag, _ = processor.sobel()
        processor.display(sobel_mag, "Sobel Edges")

        canny_edges = processor.canny()
        processor.display(canny_edges, "Full Canny Edges")


if __name__ == "__main__":
    Demo.run()
